# 1. Object Detection
Object Detection(객체 탐지)은 이미지나 영상에서 특정 객체의 존재 여부를 확인하고, 해당 객체의 위치를 바운딩 박스(bounding box)로 표시하는 기술입니다. 이는 컴퓨터 비전에서 중요한 분야로, 이미지 내에서 여러 개의 객체를 동시에 탐지하고 분류할 수 있습니다. Object Detection은 주로 딥러닝 기반의 CNN(합성곱 신경망) 모델을 활용하며, 대표적인 알고리즘으로는 R-CNN 계열(Faster R-CNN, Mask R-CNN), YOLO(You Only Look Once), SSD(Single Shot MultiBox Detector) 등이 있습니다. 이러한 기술은 자율 주행, 보안 감시, 의료 영상 분석, 증강 현실 등 다양한 분야에서 활용됩니다.

## Object Detection 논문 추천 리스트

![논문추천리스트](./images/논문추천리스트.png)

### 1. 객체 탐지의 출력
- 클래스: 사람, 자동차, 개 등
- 바운딩 박스: 객체의 위치와 크기
- confidence score: 해당 예측을 모델이 얼마나 확신하는지 나타내는 점수

> 객체 탐지는 이미지 전체에 하나의 클래스만 붙이는 Image Classification과 다름. 한 이미지 안에 여러 객체가 있어도 각각 따로 찾아낼 수 있음.

### 2. 대표적인 딥러닝 기반 계열
1. Two-Stage: Faster R-CNN, Mask R-CNN 등
    - 먼저 객체 후보 영역을 만들고, 그 후보를 다시 분류하고 박스를 보정함. 일반적으로 정밀한 탐지에 강점이 있음
2. One-Stage: YOLO, SSD, RetinaNet 등
    - 한 번의 네트워크 흐름에서 객체 클래스와 위치를 직접 예측. 일반적으로 빠르며 실시간 응용에 많이 사용

> 실무에서는 자율주행, CCTV 분석, 제조 불량 검출, 의료영상 분석 등에서 사용.  
> 다만 현대 모델에서는 "One-Stage는 항상 부정확하고 Two-Stage는 항상 정확하다"라고 단정하면 안 됨. 데이터셋, 모델 크기, 입력 해상동, 학습 방식에 따라 성능 관계는 달라짐.

# **2. YOLO**
YOLO(You Only Look Once)는 이미지를 격자 단위로 나누어 각 격자에서 객체의 위치(바운딩 박스 좌표)와 클래스를 동시에 예측하는 1단계 객체 탐지(One-Stage Detection) 알고리즘입니다. (최신 YOLO 계열은 단순히 격자로 나눔이 아닌 다중 스케일(feature map)의 여러 위치에서 객체 후보를 동시에 예측하는 방법을 사용)
한 번의 신경망 추론으로 전체 이미지의 탐지를 수행하기 때문에 매우 빠르고, 실시간 객체 탐지에 적합하다는 장점이 있습니다. YOLO 계열은 v1에서 시작해 v3까지는 Joseph Redmon이 개발했으며, 이후 v4·v7은 커뮤니티 연구자들, v5·v8·v11은 Ultralytics가 주도적으로 발전시켜 현재까지 이어지고 있습니다. 이러한 발전 과정을 거치면서 YOLO는 속도와 정확도의 균형을 잡은 대표적인 객체 탐지 모델로 자리 잡아 다양한 산업 현장에서 활용되고 있습니다.

 

### 1. Ultralytics 모델 크기 표기

- n(nano): 가장 가볍고 빠름. 실시간 실험에 유리
- s(small): 속도와 정확도의 균형이 좋아서 실무 프로토타입에 적합
- m(medium): 더 높은 정확도를 기대할 수 있지만 연산량 증가
- l(large): 정확도 중심
- x(extra-large): 가장 무겁고 높은 연산 비용

> 모델이 커지면 무조건 좋은 것은 아님. 실제 프로젝트에서는 GPU, FPS, 데이터 크기, 목표 정확도를 함께 고려해야 함

### 2. Ultralytics 주요 Task
- Detect: 객체 위지를 axis-aligned bounding box로 탐지
- Segment: 객체 별 mask를 예측하는 instance segmentation
- Pose: keypoint를 예측하는 pose estimation. 기본 pretrained 모델은 사람 관절 추정에 많이 사용되지만 custom keypoint 데이터도 학습 가능
- classify: 이미지 전체를 하나의 클래스 또는 설정된 분류 문제로 예측
- OBB(Oriented Bounding Box): 객체의 방향까지 포함한 ㅎ회전된 바운딩 박스를 예측 (박스 자체에 angle이 포함)


In [2]:
import sys
import torch
from pathlib import Path


In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.13.0+cu132
True


In [3]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.xpu.is_available():
    DEVICE = torch.device('xpu')
else:
    DEVICE = torch.device('cpu')

print(DEVICE)

cuda


In [3]:
PROJECT_ROOT = Path.cwd()
print('프로젝트 경로: ', PROJECT_ROOT)

프로젝트 경로:  c:\Users\jmsya\ai_1900_jms\4_MULTIMODAL


# **3. PascalVOC 2007**

PascalVOC 2007은 객체 탐지(Object Detection), 분할(Segmentation), 동작 인식(Action Recognition) 등의 다양한 컴퓨터 비전 과제를 위한 벤치마크 데이터셋입니다. 총 20개의 객체 클래스(예: 사람, 자동차, 개, 고양이 등)를 포함하며, 훈련(train), 검증(val), 테스트(test) 세트로 구성되어 있습니다. 각 이미지에는 객체의 경계 상자(Bounding Box) 및 해당 클래스 레이블이 주어지며, Mean Average Precision(mAP) 평가 기준을 적용하여 성능을 측정합니다.

In [4]:
VOC_ROOT = PROJECT_ROOT / 'VOCdevkit' / 'VOC2007'
YOLO_ROOT = PROJECT_ROOT / 'datasets' / 'voc_yolo'

ANNOTATIONS_DIR = VOC_ROOT / 'Annotations'
IMAGES_DIR = VOC_ROOT / 'JPEGImages'
IMAGESETS_DIR = VOC_ROOT / 'ImageSets' / 'Main'

print('VOC_ROOT: ', VOC_ROOT)
print('YOLO_ROOT: ', YOLO_ROOT)

VOC_ROOT:  c:\Users\jmsya\ai_1900_jms\4_MULTIMODAL\VOCdevkit\VOC2007
YOLO_ROOT:  c:\Users\jmsya\ai_1900_jms\4_MULTIMODAL\datasets\voc_yolo


# **4. YOLO annotation 기본 구조**

```
예: 6 0.189800 0.812057 0.379440 0.373544
<class_id> <x_center> <y_center> <width> <height>
```
- class_id: 0부터 시작하는 클래스 번호
- x_center, y_center: 박스 중심점
- width, height: 박스의 너비와 높이
- 좌표는 모두 이미지 크기로 나눈 0~1사이 값

> VOC XML의 (xmin, ymin, xmax, ymax)를 YOLO 형식으로 바꿀 때는 중심점과 크기를 계산한 뒤 이미지 너비와 높이로 정규화해야 함.

In [4]:
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

CLASS_TO_ID = {name: idx for idx, name in enumerate(VOC_CLASSES)}
CLASS_TO_ID

{'aeroplane': 0,
 'bicycle': 1,
 'bird': 2,
 'boat': 3,
 'bottle': 4,
 'bus': 5,
 'car': 6,
 'cat': 7,
 'chair': 8,
 'cow': 9,
 'diningtable': 10,
 'dog': 11,
 'horse': 12,
 'motorbike': 13,
 'person': 14,
 'pottedplant': 15,
 'sheep': 16,
 'sofa': 17,
 'train': 18,
 'tvmonitor': 19}

In [5]:
def voc_box_to_yolo(xmin, ymin, xmax, ymax, image_w, image_h):
    x_center = ((xmin + xmax)/2.0) / image_w
    y_center = ((ymin + ymax)/2.0) / image_h
    box_w = (xmax - xmin)/image_w
    box_h = (ymax - ymin)/image_h
    return x_center, y_center, box_w, box_w

In [6]:
import xml.etree.ElementTree as ET

In [ ]:
def convert_voc_xml(xml_path: Path, label_path: Path):
    root = ET.parse(xml_path).getroot()
    size = root.find('size')
    image_w = float(size.findtext('width'))
    image_h = float(size.findtext('height'))
    
    lines =[]
    for obj in root.findall('object'):
        class_name = obj.findtext('name')
        if class_name not in CLASS_TO_ID:
            continue
        
        difficult = int(obj.findtext('difficult', default='0'))
        if difficult == 1:
            continue
        
        bnd = obj.find('bndbox')
    